In [1]:
import time
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, ConcatDataset
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import EarlyStopping
from lightning.pytorch.loggers import TensorBoardLogger

from data import SyntheticLogReturnsDataset, generate_datasets
from model import FinancialLSTMModule

## Config

In [2]:
config = {
    'N_DATASET_TRAIN': 70,
    'N_DATASET_VAL': 20,
    'N_DATASET_TEST': 10,
    'N_STOCK': 8,
    'T': 500,
    'DISTRIBUTION': 'normal',
    'SEED': 100,
    
    'WINDOW': 60,
    'HIDDEN_SIZE': 64,
    'NUM_LAYERS': 2,
    'DROPOUT': 0.2,
    'TASK': 'reconstruction',
    'LEARNING_RATE': 1e-4,

    'NUM_EPOCHS': 10,
    'BATCH_SIZE': 16,
    'SHUFFLE_BATCHES': True,
    
    'PATIENCE': 3,
    'MIN_DELTA': 1e-6
}

## Datasets and DataLoaders

In [3]:
train_datasets, val_datasets, test_datasets = generate_datasets(config)

train_loader = DataLoader(ConcatDataset(train_datasets), batch_size=config['BATCH_SIZE'], shuffle=config['SHUFFLE_BATCHES'])
val_loader   = DataLoader(ConcatDataset(val_datasets),   batch_size=config['BATCH_SIZE'], shuffle=False)
test_loader  = DataLoader(ConcatDataset(test_datasets),  batch_size=1, shuffle=False)

## Model and Trainer

In [ ]:
model = FinancialLSTMModule(
    window      = config['WINDOW'],
    hidden_size = config['HIDDEN_SIZE'],
    num_layers  = config['NUM_LAYERS'],
    dropout     = config['DROPOUT'],
    task        = config['TASK'],
    lr          = config['LEARNING_RATE']
)

# TODO add early stop conditionally, also add ModelCheckpoint callback
early_stop_callback = EarlyStopping(
    monitor   = 'val_loss',
    min_delta = config['MIN_DELTA'],
    patience  = config['PATIENCE'],
    verbose   = False,
    mode      = 'min'
)

tb_logger = TensorBoardLogger(
    save_dir          = 'logs/',
    name              = 'financial_lstm',
    version           = 'estimation_run_short',
    default_hp_metric = False
)

trainer = Trainer(
    max_epochs = config['NUM_EPOCHS'],
    callbacks  = [early_stop_callback],
    logger     = tb_logger
)

## Train model

In [ ]:
start_time = time.time()

trainer.fit(model, train_loader, val_loader)

print(f"Done! Trained model in {time.time() - start_time:>7f} seconds!")

## Resume training

In [ ]:
# TODO Test if it works

model = FinancialLSTMModule()
trainer = Trainer()

trainer.fit(model, train_loader, val_loader, ckpt_path='/path/to/checkpoint.ckpt')

## Load model

In [29]:
model = FinancialLSTMModule.load_from_checkpoint('/path/to/checkpoint.ckpt')

In [30]:
# For additional logging

tb_logger = TensorBoardLogger(
    save_dir          = 'logs/',
    name              = 'financial_lstm',
    version           = 'estimation_batch=1',
    default_hp_metric = False
)

# Benchmark

In [31]:
from scipy import stats

def OLS_on_sample(stock_returns, r_market, r_market_next=None):
    beta, alpha, _, _, _ = stats.linregress(r_market, stock_returns)
    
    if r_market_next is None:
        reconstruction = alpha + beta * r_market
    else:
        reconstruction = alpha + beta * r_market_next.squeeze(-1)

    return reconstruction, alpha, beta

In [32]:
true_alphas = torch.stack([test_dataset.alphas for test_dataset in test_datasets])
true_betas  = torch.stack([test_dataset.betas for test_dataset in test_datasets])

loss_function = torch.nn.MSELoss(reduction='mean')

model_summary = {
    'recon_loss': [],
    'alpha_loss': [],
    'beta_loss':[],
    'alpha': [],
    'beta': []
}

ols_summary = {
    'recon_loss': [],
    'alpha_loss': [],
    'beta_loss':[],
    'alpha': [],
    'beta': []
}

n_stock = config['N_STOCK']
len_dataset = n_stock * config['T']

start_time = time.time()
print("Evauating model on test set...")

model.eval()
with torch.no_grad():
    for i, data in enumerate(test_loader):
        curr_dataset = i // len_dataset
        curr_stock = i % n_stock

        true_alpha = true_alphas[curr_dataset, curr_stock]
        true_beta  = true_betas[curr_dataset, curr_stock]

        stock_returns = data['stock_returns']
        r_market      = data['r_market']
        target        = data['target']

        if config['TASK'] == 'prediction':
            r_market_next = data['r_market_next']
        else:
            r_market_next = None
        
        # Model loss
        reconstruction, alpha, beta = model(stock_returns, r_market, r_market_next)
        model_summary['recon_loss'].append(loss_function(reconstruction, target).item())
        model_summary['alpha_loss'].append(loss_function(alpha, true_alpha).item())
        model_summary['beta_loss'].append(loss_function(beta, true_beta).item())

        model_summary['alpha'].append(alpha.detach())
        model_summary['beta'].append(beta.detach())

        
        #OLS and OLS loss
        stock_returns = stock_returns.squeeze()
        r_market      = r_market.squeeze()
        target        = target.squeeze()

        if config['TASK'] == 'prediction':
            r_market_next = r_market_next.squeeze()
        
        reconstruction, alpha, beta = OLS_on_sample(stock_returns, r_market, r_market_next)
        alpha, beta = torch.Tensor([alpha]), torch.Tensor([beta])
        ols_summary['recon_loss'].append(loss_function(reconstruction, target).item())
        ols_summary['alpha_loss'].append(loss_function(alpha, true_alpha).item())
        ols_summary['beta_loss'].append(loss_function(beta, true_beta).item())

        ols_summary['alpha'].append(alpha)
        ols_summary['beta'].append(beta)

model_summary['alpha'] = torch.Tensor(model_summary['alpha']).reshape(config['N_DATASET_TEST'], config['T'], config['N_STOCK'])
model_summary['beta'] = torch.Tensor(model_summary['beta']).reshape(config['N_DATASET_TEST'], config['T'], config['N_STOCK'])

ols_summary['alpha'] = torch.Tensor(ols_summary['alpha']).reshape(config['N_DATASET_TEST'], config['T'], config['N_STOCK'])
ols_summary['beta'] = torch.Tensor(ols_summary['beta']).reshape(config['N_DATASET_TEST'], config['T'], config['N_STOCK'])

print(f"Done! time={time.time() - start_time:>7f} seconds")

Evauating model on test set...
Done! time=74.771325 seconds


In [33]:
# Calculate test metrics for hp_metric
model_test_loss = np.mean(model_summary['recon_loss'])
ols_test_loss = np.mean(ols_summary['recon_loss'])

# Log hp_metric (model_test_loss)
tb_logger.log_hyperparams(
    params=config,
    metrics={'hp_metric': model_test_loss}
)

In [34]:
plt.figure(figsize=(20, 10))

plt.scatter(model_summary['recon_loss'], ols_summary['recon_loss'], marker='.')
plt.xlabel('Model Loss')
plt.ylabel('OLS Loss')

identity = np.linspace(min(ols_summary['recon_loss']), max(ols_summary['recon_loss']), 1000)
plt.plot(identity, identity, 'r--')

plt.title('Model vs OLS Reconstruction loss (MSE)')
plt.grid(alpha=0.5)

# Log to TensorBoard
tb_logger.experiment.add_figure('Model vs OLS Reconstruction loss (MSE)', plt.gcf())
plt.show()

In [35]:
plt.figure(figsize=(20, 10))

plt.hist(model_summary['recon_loss'], bins=100, alpha=0.6, label='Model')
plt.hist(ols_summary['recon_loss'], bins=100, alpha=0.6, label='OLS')

plt.axvline(model_test_loss, color='blue', linestyle='--', label=f'Avg Model Loss ({model_test_loss:.7f})')
plt.axvline(ols_test_loss, color='orange', linestyle='--', label=f'Avg OLS Loss    ({ols_test_loss:.7f})')

plt.title('Model vs OLS Reconstruction loss (MSE)')
plt.grid(alpha=0.5)
plt.legend()

# Log to TensorBoard
tb_logger.experiment.add_figure('Model vs OLS Reconstruction loss (MSE) Histogram', plt.gcf())

In [36]:
model_alpha_test_loss = np.mean(model_summary['alpha_loss'])
ols_alpha_test_loss = np.mean(ols_summary['alpha_loss'])

plt.figure(figsize=(20, 10))

plt.hist(model_summary['alpha_loss'], bins=100, alpha=0.6, label='Model')
plt.hist(ols_summary['alpha_loss'], bins=100, alpha=0.6, label='OLS')

plt.axvline(model_alpha_test_loss, color='blue', linestyle='--', label=f'Avg Model Loss ({model_alpha_test_loss:.7f})')
plt.axvline(ols_alpha_test_loss, color='orange', linestyle='--', label=f'Avg OLS Loss    ({ols_alpha_test_loss:.7f})')

plt.title('Model vs OLS Alpha loss (MSE)')
plt.grid(alpha=0.5)
plt.legend()

# Log to TensorBoard
tb_logger.experiment.add_figure('Model vs OLS Alpha loss (MSE)', plt.gcf())

In [37]:
model_beta_test_loss = np.mean(model_summary['beta_loss'])
ols_beta_test_loss = np.mean(ols_summary['beta_loss'])

plt.figure(figsize=(20, 10))

plt.hist(model_summary['beta_loss'], bins=100, alpha=0.6, label='Model')
plt.hist(ols_summary['beta_loss'], bins=100, alpha=0.6, label='OLS')

plt.axvline(model_beta_test_loss, color='blue', linestyle='--', label=f'Avg Model Loss ({model_beta_test_loss:.4f})')
plt.axvline(ols_beta_test_loss, color='orange', linestyle='--', label=f'Avg OLS Loss    ({ols_beta_test_loss:.4f})')

plt.title('Model vs OLS Beta loss (MSE)')
plt.grid(alpha=0.5)
plt.legend()

# Log to TensorBoard
tb_logger.experiment.add_figure('Model vs OLS Beta loss (MSE)', plt.gcf())

In [38]:
tb_logger.experiment.add_scalar('test/model_alpha_loss', model_alpha_test_loss)
tb_logger.experiment.add_scalar('test/model_beta_loss', model_beta_test_loss)
idxs = list(range(16))

In [39]:
fig, axs = plt.subplots(4, 4, figsize=(20, 20))
fig.suptitle('Model vs OLS Alpha estimation')

axs = axs.flatten()

for ax, idx in zip(axs, idxs):
    stock_idx   = idx % config['N_STOCK']
    dataset_idx = idx // config['N_DATASET_TEST']
    
    true_alpha  = true_alphas[dataset_idx, stock_idx]
    model_alpha = model_summary['alpha'][dataset_idx, :, stock_idx]
    ols_alpha   = ols_summary['alpha'][dataset_idx, :, stock_idx]

    sample = np.arange(len(model_alpha))

    ax.set_title(f'Stock {idx}')
    ax.scatter(sample, model_alpha, label='Model', marker='.')
    ax.scatter(sample, ols_alpha, label='OLS', marker='.')
    ax.axhline(true_alpha, color='magenta', linestyle='--', label='True alpha')

    ax.legend()
    ax.grid(alpha=0.5)

# Log to TensorBoard
tb_logger.experiment.add_figure('Model vs OLS Alpha estimation', fig)

In [40]:
fig, axs = plt.subplots(4, 4, figsize=(20, 20))
fig.suptitle('Model vs OLS Beta estimation')

axs = axs.flatten()

for ax, idx in zip(axs, idxs):
    stock_idx   = idx % config['N_STOCK']
    dataset_idx = idx // config['N_DATASET_TEST']
    
    true_beta  = true_betas[dataset_idx, stock_idx]
    model_beta = model_summary['beta'][dataset_idx, :, stock_idx]
    ols_beta   = ols_summary['beta'][dataset_idx, :, stock_idx]

    sample = np.arange(len(model_beta))

    ax.set_title(f'Stock {idx}')
    ax.scatter(sample, model_beta, label='Model', marker='.')
    ax.scatter(sample, ols_beta, label='OLS', marker='.')
    ax.axhline(true_beta, color='magenta', linestyle='--', label='True beta')

    ax.legend()
    ax.grid(alpha=0.5)

# Log to TensorBoard
tb_logger.experiment.add_figure('Model vs OLS Beta estimation', fig)

In [41]:
tb_logger.save()